# Python practice / Thực hành Python

Tôi lưu bài này trong nhật ký học để ôn lại cách viết và đọc code.

**Nguồn / Source:** Bản Colab `problem_12_best_auto.ipynb`. Các bài có tên “Bản sao” là tài liệu thực hành tham khảo; tên file không được xem là bằng chứng tác giả.

**Cách dùng / Run:** chạy các cell theo thứ tự. Thư viện được dùng: `numpy`, `pandas`, `scipy`, `scipy.spatial.distance`, `tqdm`, `typing`.

**Execution:** ARCHIVED_SOURCE — bản mã học tập, chưa ghi nhận chạy lại đầy đủ trong đợt này. Output lịch sử được bỏ; hướng dẫn và lỗi cần xử lý nằm trong bảng kỹ thuật.

Learning archive: coursework/tutorial practice, with original code retained for study. Original tutorial references embedded in the source are retained below.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from scipy.spatial.distance import cosine
from typing import List, Dict, Tuple

In [ ]:
def load_vocab_dict(path: str) -> Tuple[List[str], Dict[str, int]]:
    """
    Reads a vocabulary list from a file and creates a dictionary mapping each word to its index.

    Args:
        path (str): The file path to the vocabulary list.

    Returns:
        Tuple[List[str], Dict[str, int]]: A tuple containing the list of vocabulary words and a dictionary
                                          mapping each word to its index.
    """
    vocab = open(path).read().strip().split('\n')
    return vocab, {word: idx for idx, word in enumerate(vocab)}

In [ ]:
def read_corpus(path: str) -> List[str]:
    """Reads the corpus from a file, excluding the last empty entry if the file ends with a newline.

    Args:
        path (str): The file path to the corpus.

    Returns:
        List[str]: A list of strings, each representing a line from the file.
    """
    return open(path).read().strip().split('\n')

In [ ]:
def counting(corpus: List[str], V: List[str], V_C: List[str], V_set: Dict[str, int], V_C_set: Dict[str, int]) -> np.ndarray:
    """
    Generates a co-occurrence (counting) matrix from the given corpus, considering specified vocabularies and a window size.

    Args:
        corpus (List[str]): The corpus as a list of sentences.
        V (List[str]): The list of vocabulary words.
        V_C (List[str]): The list of context vocabulary words.
        V_set (Dict[str, int]): A dictionary mapping vocabulary words to their indices.
        V_C_set (Dict[str, int]): A dictionary mapping context vocabulary words to their indices.

    Returns:
        np.ndarray: A 2D NumPy array representing the co-occurrence matrix with dimensions (len(V), len(V_C)).
    """
    # Initialize the matrix to hold word vectors
    C = np.zeros((len(V), len(V_C)), dtype=float)

    for line in tqdm(corpus): # Iterate over each word in the original dataset
        # Append start and end tokens to the sentence
        words = ['<s>'] + line.split(' ') + ['</s>']
        length = len(words)

        for idx, word in enumerate(words): # Iterate over each word in the current sentence
            # Skip '<s>' and '</s>', as they are not real words
            if idx > 0 and idx < length - 1 and word in V_set:
                ### BEGIN SOLUTION
                # Window ±2 with 1/dist weighting + light frequency debias for contexts
                # Build (and cache) a frequency map for context words to downweight very common ones
                if not hasattr(counting, "_ctx_freq"):
                    from collections import Counter
                    counting._ctx_freq = Counter()
                    for line_ in corpus:
                        counting._ctx_freq.update(line_.strip().split())
                ctx_freq = counting._ctx_freq

                import math
                w_idx = V_set[word]
                window_size = 2
                for dist in range(1, window_size + 1):
                    # left
                    lp = idx - dist
                    if lp >= 0:
                        c = words[lp]
                        if c in V_C_set:
                            c_idx = V_C_set[c]
                            weight = (1.0 / dist) * (1.0 / (1.0 + math.log1p(ctx_freq.get(c, 1))))
                            C[w_idx, c_idx] += weight
                    # right
                    rp = idx + dist
                    if rp < length:
                        c = words[rp]
                        if c in V_C_set:
                            c_idx = V_C_set[c]
                            weight = (1.0 / dist) * (1.0 / (1.0 + math.log1p(ctx_freq.get(c, 1))))
                            C[w_idx, c_idx] += weight
                            ### END SOLUTION
    return C

In [ ]:
def eval_word_similarity(C: np.ndarray, V_set: Dict[str, int], path: str) -> float:
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f.readlines()[1:]:
            parts = line.strip().split('\t')
            if len(parts) != 3:
                continue
            w1, w2, score_str = parts
            if w1 in V_set and w2 in V_set:
                score = float(score_str)
                v1, v2 = C[V_set[w1]], C[V_set[w2]]
                sim = 1 - cosine(v1, v2) if np.linalg.norm(v1) > 0 and np.linalg.norm(v2) > 0 else 0.0
                pairs.append((sim, score))

    if not pairs:
        return 0.0

    x, y = zip(*pairs)
    corr = stats.spearmanr(x, y).correlation
    return float(corr) if not np.isnan(corr) else 0.0

In [ ]:
# Read the main vocabulary and its indices from a file,
# creating a list of words (V) and a dictionary mapping words to indices (V_set).
V, V_set = load_vocab_dict('./data/main_words.txt')

# Read the context vocabulary and its indices from a separate file,
# creating a list of context words (V_C) and a dictionary mapping these words to indices (V_C_set).
V_C, V_C_set = load_vocab_dict('./data/context_words.txt')

# Read the corpus from a text file, creating a list where each item represents a document or line in the corpus.
corpus = read_corpus('./data/corpus.txt')

# Generate a co-occurrence (counting) matrix from the corpus using the main and context vocabularies.
C = counting(corpus, V, V_C, V_set, V_C_set)

In [ ]:
### BEGIN PUBLIC TESTS
eval_word_similarity(C, V_set, './data/men.txt')
### BEGIN PUBLIC TESTS

In [ ]:
### BEGIN PUBLIC TESTS
eval_word_similarity(C, V_set, './data/simlex-999.txt')
### BEGIN PUBLIC TESTS

In [ ]:
def improve_C(C: np.ndarray, corpus: List[str], V_C: List[str], V_C_set: Dict[str, int]) -> np.ndarray:
    ### BEGIN SOLUTION
    import numpy as np

    # Helper: safe L2 normalize rows
    def _l2norm(X):
        n = np.linalg.norm(X, axis=1, keepdims=True)
        n[n == 0.0] = 1.0
        return X / n

    # Helper: sqrt (power) normalization to reduce burstiness
    def _sqrt_norm(X):
        X[X < 0] = 0.0
        return np.sqrt(X, dtype=X.dtype)

    # Helper: Truncated SVD with sklearn if available, else numpy fallback
    def _svd(X, k):
        try:
            from sklearn.decomposition import TruncatedSVD
            svd = TruncatedSVD(n_components=min(k, max(2, min(X.shape)-1)), random_state=0)
            Z = svd.fit_transform(X)
            return Z
        except Exception:
            # numpy fallback (dense) — keep memory in mind
            try:
                U, S, VT = np.linalg.svd(X, full_matrices=False)
                kk = min(k, U.shape[1])
                return U[:, :kk] * S[:kk]
            except Exception:
                # If SVD fails, return X as-is
                return X

    # Compute marginals from current C (counts/weighted counts)
    V_size = C.shape[0]
    V_C_size = C.shape[1]
    total = float(C.sum()) + 1e-12

    row_sums = C.sum(axis=1, keepdims=True)
    col_sums = C.sum(axis=0, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    col_sums[col_sums == 0] = 1.0

    # Variant A: PPMI vanilla
    A = np.zeros_like(C)
    for i_ in range(V_size):
        p_w = row_sums[i_, 0] / total
        for j_ in range(V_C_size):
            p_wc = C[i_, j_] / total
            if p_wc <= 0.0:
                continue
            p_c = col_sums[0, j_] / total
            if p_w > 0 and p_c > 0:
                pmi = np.log2(p_wc / (p_w * p_c))
                if pmi > 0:
                    A[i_, j_] = pmi

    # Variant B: PPMI with context smoothing (alpha=0.75)
    alpha = 0.75
    B = np.zeros_like(C)
    col_sums_alpha = col_sums ** alpha
    col_sums_alpha[col_sums_alpha == 0] = 1.0
    for i_ in range(V_size):
        p_w = row_sums[i_, 0] / total
        for j_ in range(V_C_size):
            p_wc = C[i_, j_] / total
            if p_wc <= 0.0:
                continue
            p_c = (col_sums_alpha[0, j_] / (total ** alpha))
            if p_w > 0 and p_c > 0:
                pmi = np.log2(p_wc / (p_w * p_c))
                if pmi > 0:
                    B[i_, j_] = pmi

    # Variant C: SPPMI with small shift (k=3) + context smoothing
    k = 3.0
    Csh = np.zeros_like(C)
    for i_ in range(V_size):
        p_w = row_sums[i_, 0] / total
        for j_ in range(V_C_size):
            p_wc = C[i_, j_] / total
            if p_wc <= 0.0:
                continue
            p_c = (col_sums_alpha[0, j_] / (total ** alpha))
            if p_w > 0 and p_c > 0:
                pmi = np.log2(p_wc / (p_w * p_c))
                sppmi = pmi - np.log2(k)
                if sppmi > 0:
                    Csh[i_, j_] = sppmi

    # Power-normalize before SVD
    A_s = _sqrt_norm(A.copy())
    B_s = _sqrt_norm(B.copy())
    Csh_s = _sqrt_norm(Csh.copy())

    # Build projected features
    A100 = _svd(A_s, 100)
    B100 = _svd(B_s, 100)
    C50  = _svd(Csh_s, 50)

    # Concatenate multi-view features
    feats = []
    for Z in (A100, B100, C50):
        Z = _l2norm(Z.astype(np.float32, copy=False))
        feats.append(Z)
    X = np.concatenate(feats, axis=1) if len(feats) > 1 else feats[0]

    # Final L2
    X = _l2norm(X.astype(np.float32, copy=False))
    return X
    ### END SOLUTION

## Ghi chú học tập / Study prompts

- Đầu vào của từng hàm là gì, đầu ra có kiểu và kích thước nào?
- Thay đổi tham số hoặc dữ liệu nhỏ sẽ ảnh hưởng kết quả ra sao?
- Cell nào cần dữ liệu bên ngoài, thư viện hoặc thao tác nhập tay?

Explain the data flow, modify one example, and record the result when revisiting this exercise.